In [1]:
#!/usr/bin/env python
# coding: utf-8
import torch, json, random, time 
from pathlib import Path
from typing import Iterable, List
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, PreTrainedTokenizerFast
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from tokenizers.normalizers import Sequence, NFC
from huggingface_hub import hf_hub_download


In [ ]:
# authenticate if needed 
#from huggingface_hub import login
# This will prompt you for the token once and save it to ~/.cache/huggingface/token
#login()

# STEP 1 : TRAIN THE TOKENIZER 

## 1. set fixed parameters 

In [51]:
# --- Configuration ---
MODEL_ID            = "google/gemma-3-270m"
OUT_DIR_Tokenizer   = Path("./gemma3-270M-kin-en-64k_tokenizer")
OUT_DIR_NewModel    = Path("./gemma3-270M-kin-en-64k_model")


VOCAB_SIZE      = 64_000          # vocabulary size of 64k 
MAX_TRAIN_TEXTS = 150_000          # Total docs (mix of Kin + En)
SEED            = 1011

# Language balance for tokenizer training
KIN_RATIO       = 0.70            # 70% Kinyarwanda
EN_RATIO        = 0.30            # 30% English


torch.manual_seed(SEED)
random.seed(SEED)
OUT_DIR_Tokenizer.mkdir(parents=True, exist_ok=True)
OUT_DIR_NewModel.mkdir(parents=True, exist_ok=True) 


## 2. Import model and Extracting the Embedding Matrix

The embedding matrix is a massive lookup table. Its shape is always (Vocabulary_Size, Hidden_Dimension). We need to pull this matrix out so we can copy its weights later.

In [3]:
xmodel_id = "google/gemma-3-270m"

print("[load] Original model & tokenizer")
tokenizer_orig = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
model_orig     = AutoModelForCausalLM.from_pretrained(MODEL_ID, trust_remote_code=True)

[load] Original model & tokenizer


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

In [6]:
# get the vocabulary 
vocab = tokenizer_orig.get_vocab()
list_vocab = list(vocab.keys())

# embedding matrix 
E0 = model_orig.get_input_embeddings().weight.detach().cpu()

# Get the dimensions
V_old = E0.shape[0]   # Old vocabulary size
hidden = E0.shape[1]  # The dimension of the vector representing each token
    

print(f"length Vocabulary list: {len(list_vocab)} tokens")
print(f"Original Vocabulary Size (V_old): {V_old} tokens")
print(f"Hidden Dimension (Embedding size): {hidden}")
print(f"Embedding Matrix Shape: {E0.shape}")




length Vocabulary list: 262145 tokens
Original Vocabulary Size (V_old): 262144 tokens
Hidden Dimension (Embedding size): 640
Embedding Matrix Shape: torch.Size([262144, 640])


## 2. Playing with the Tokenizer (Encoding & Embeddings)

Before we replace the tokenizer, let's see why we need to! We will grab a random token to see its mathematical representation (embedding), and then encode phrases in English, French, and Kinyarwanda.

Notice how the English and French phrases are broken into neat, whole-word tokens, while the Kinyarwanda phrase is likely shattered into many tiny, inefficient fragments.

In [9]:


# --- 1. Inspecting a Random Token's Embedding ---
random_token = random.choice(list_vocab)
token_id = vocab[random_token]

# Get the embedding layer
embeddings = model_orig.get_input_embeddings().weight.detach().cpu()
token_embedding = embeddings[token_id]

print(f"--- Random Token Inspection ---")
print(f"Token string: '{random_token}'")
print(f"Token ID: {token_id}")
print(f"Embedding Vector Shape: {token_embedding.shape} (This is the 'meaning' vector)")
# FIX: We must cast to float32 before calling .numpy() because NumPy doesn't support bfloat16!
print(f"First 5 values of the vector: {token_embedding[:5].to(torch.float32).numpy()}\n")

## for example token id 234118 is for 'ബ്രുവരി'


--- Random Token Inspection ---
Token string: '▁sydney'
Token ID: 236137
Embedding Vector Shape: torch.Size([640]) (This is the 'meaning' vector)
First 5 values of the vector: [ 0.02490234  0.00878906 -0.04956055 -0.00308228 -0.06787109]



In [10]:
# --- 2. Encoding Multilingual Phrases ---
phrases = {
    "English": "Hello, how are you doing today?",
    "French": "Bonjour, comment allez-vous aujourd'hui ?",
    "Kinyarwanda": "Muraho, amakuru yawe uyumunsi?"
}

print(f"--- Tokenization Comparison ---")
for lang, phrase in phrases.items():
    # Encode turns text into a list of integers (token IDs)
    token_ids = tokenizer_orig.encode(phrase)

    # Let's decode them one by one to see the exact chunks
    chunks = [tokenizer_orig.decode([tid]) for tid in token_ids]

    print(f"[{lang}]")
    print(f"  Text:   {phrase}")
    print(f"  Length: {len(token_ids)} tokens")
    print(f"  IDs:    {token_ids}")
    print(f"  Chunks: {chunks}\n")

--- Tokenization Comparison ---
[English]
  Text:   Hello, how are you doing today?
  Length: 9 tokens
  IDs:    [2, 9259, 236764, 1217, 659, 611, 3490, 3124, 236881]
  Chunks: ['<bos>', 'Hello', ',', ' how', ' are', ' you', ' doing', ' today', '?']

[French]
  Text:   Bonjour, comment allez-vous aujourd'hui ?
  Length: 11 tokens
  IDs:    [2, 53406, 236764, 5739, 99219, 236772, 23841, 38014, 236789, 25747, 2360]
  Chunks: ['<bos>', 'Bonjour', ',', ' comment', ' allez', '-', 'vous', ' aujourd', "'", 'hui', ' ?']

[Kinyarwanda]
  Text:   Muraho, amakuru yawe uyumunsi?
  Length: 14 tokens
  IDs:    [2, 39151, 21780, 236764, 1006, 628, 14406, 4334, 977, 38363, 597, 602, 5399, 236881]
  Chunks: ['<bos>', 'Mur', 'aho', ',', ' am', 'ak', 'uru', ' ya', 'we', ' uy', 'um', 'un', 'si', '?']



## 3. Extracting Special Tokens

This is a critical step. Tokenizers use special tokens to mark the beginning of text, ends of turns in a chat, padding, etc.
 Sometimes, the Hugging Face API doesn't expose all of these perfectly through python properties (like orig_tok.eos_token). To be safe, your code brilliantly reads the raw tokenizer.json file directly from the Hugging Face hub to find everything marked as "special": true.


In [16]:
print("[specials] Extracting special tokens...")

# 1. Download and read the raw tokenizer.json file
tokenizer_file_path = hf_hub_download(xmodel_id, "tokenizer.json")
tok_json = json.loads(Path(tokenizer_file_path).read_text(encoding="utf-8"))

specials: List[str] = []

# 2. Look through the raw JSON for any added token marked as special
added_tokens = tok_json.get("added_tokens", [])
if added_tokens:
    for at in added_tokens:
        if isinstance(at, dict) and at.get("special", False) and "content" in at:
            specials.append(at["content"])

# 3. Ensure core specials from the standard Hugging Face API are also included
for nm in ("bos_token", "eos_token", "pad_token", "unk_token"):
    s = getattr(tokenizer_orig, nm, None)
    # If the token exists and isn't already in our list, add it
    if isinstance(s, str) and s not in specials:
        specials.append(s)

# 4. Clean up the list (remove duplicates and sort)
specials = sorted(set(specials))

print(f"Found {len(specials)} special tokens that MUST be preserved:")
for idx, st in enumerate(specials):
    print(f"  {idx + 1}: {st}")


[specials] Extracting special tokens...
Found 7 special tokens that MUST be preserved:
  1: <bos>
  2: <end_of_turn>
  3: <eos>
  4: <image_soft_token>
  5: <pad>
  6: <start_of_turn>
  7: <unk>


## 4. Dataset preparation 



In [29]:
def iter_mixed_corpus(max_docs: int) -> Iterable[str]:
    """Stream Kinyarwanda AND English texts with controlled balance."""
    print(f"[data] Streaming mixed corpus (Kin:{KIN_RATIO*100:.0f}%, En:{EN_RATIO*100:.0f}%)...")
    
    # Load Kinyarwanda (your 1M document dataset)
    ds_kin = load_dataset("mbazaNLP/kinyarwanda_monolingual_v01.1", split="train", streaming=True)
    # Load English (FineWeb-2 subset)
    ds_en = load_dataset("HuggingFaceFW/fineweb", name="CC-MAIN-2025-26", split="train", streaming=True)
    
    kin_target = int(max_docs * KIN_RATIO)
    en_target = int(max_docs * EN_RATIO)
    
    kin_count = 0
    en_count = 0
    kin_iter = iter(ds_kin)
    en_iter = iter(ds_en)
    
    while kin_count < kin_target or en_count < en_target:
        # Alternate between languages to ensure balance
        if kin_count < kin_target:
            try:
                ex = next(kin_iter)
                txt = (ex.get("text", "") or "").strip()
                if len(txt) >= 5:
                    yield txt
                    kin_count += 1
            except StopIteration:
                print("[warn] Kinyarwanda dataset exhausted")
                kin_target = kin_count
        
        if en_count < en_target:
            try:
                ex = next(en_iter)
                txt = (ex.get("text", "") or "").strip()
                if len(txt) >= 5:
                    yield txt
                    en_count += 1
            except StopIteration:
                print("[warn] English dataset exhausted")
                en_target = en_count
    
    print(f"[data] Completed: Kin={kin_count}, En={en_count}, Total={kin_count+en_count}")

## 5. Training the Tokenizer

Here is the actual training block. We are using ByteLevel BPE. Why? Because standard BPE throws an error if it sees a character it has never seen in training. ByteLevel BPE falls back to raw bytes (0-255) for unknown characters, meaning it can technically encode any alphabet in the world, even if it has to break it down into 1-byte chunks!

In [31]:

print(f"[train] Initializing ByteLevel BPE Tokenizer...")

# 1. Initialize BPE model with byte fallback and our safe unk_token
safe_unk_token = getattr(tokenizer_orig, "unk_token", None) or "<unk>"

tok = Tokenizer(BPE(unk_token=safe_unk_token, byte_fallback=True))

# 2. Normalization: NFC handles unicode characters (e.g., standardizing accents)
tok.normalizer = Sequence([NFC()])
# 3. Pre-tokenizer and Decoder: ByteLevel treats string as bytes and maps spaces
tok.pre_tokenizer = ByteLevel(add_prefix_space=True)
tok.decoder = ByteLevelDecoder()

# 4. Initialize the Trainer
print(f"[train] Starting training on {MAX_TRAIN_TEXTS} Kinyarwanda texts... (This may take a minute)")

print('TIME START:', time.asctime())

trainer = BpeTrainer(vocab_size=VOCAB_SIZE, 
                     min_frequency=2, 
                     special_tokens=specials, 
                     show_progress=True)
tok.train_from_iterator(iter_mixed_corpus(MAX_TRAIN_TEXTS), trainer=trainer)

print("[train] Tokenizer training complete!")

print('TIME END:', time.asctime())




[train] Initializing ByteLevel BPE Tokenizer...
[train] Starting training on 150000 Kinyarwanda texts... (This may take a minute)
TIME START: Sun Mar  1 13:31:31 2026
[data] Streaming mixed corpus (Kin:70%, En:30%)...


Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/250 [00:00<?, ?it/s]

[data] Completed: Kin=105000, En=45000, Total=150000



[train] Tokenizer training complete!
TIME END: Sun Mar  1 13:32:02 2026


In [38]:
## save it 

# Wrap & Save Tokenizer
new_fast = PreTrainedTokenizerFast(tokenizer_object=tok)
m = {k: getattr(tokenizer_orig, k) for k in ["bos_token", "eos_token", "pad_token", "unk_token"] if getattr(tokenizer_orig, k)}
if m: new_fast.add_special_tokens(m)
## if the tokenizer had a chat_template you would needto add it
## this is a basemodel so we d not add it 
#if getattr(tokenizer_orig, "chat_template", None):
#    new_fast.chat_template = tokenizer_orig.chat_template
#save it in the tokenizer folder 
    
new_fast.save_pretrained(OUT_DIR_Tokenizer )

print(f"[save] Success! New Kinyarwanda tokenizer saved to: {OUT_DIR_Tokenizer}")


[save] Success! New Kinyarwanda tokenizer saved to: gemma3-270M-kin-en-64k_tokenizer


In [40]:
 #Quick Test ---
sample_text = "Muraho, amakuru yawe uyumunsi?"
print(f"\nQuick Test of the NEW Tokenizer:")
print(f"Input: {sample_text}")
encoded = new_fast.encode(sample_text)
print(f"Token IDs: {encoded}")
print(f"Chunks: {[new_fast.decode([tid]) for tid in encoded]}")



Quick Test of the NEW Tokenizer:
Input: Muraho, amakuru yawe uyumunsi?
Token IDs: [290, 5572, 18, 1762, 5405, 11283, 7124, 37]
Chunks: [' M', 'uraho', ',', ' amakuru', ' yawe', ' uy', 'umunsi', '?']


In [41]:
phrases = {
    "English": "Hello, how are you doing today?",
    "French": "Bonjour, comment allez-vous aujourd'hui ?",
    "Kinyarwanda": "Muraho, amakuru yawe uyumunsi?"
}

print(f"--- Tokenization Comparison ---")
for lang, phrase in phrases.items():
    # Encode turns text into a list of integers (token IDs)
    token_ids = new_fast.encode(phrase)

    # Let's decode them one by one to see the exact chunks
    chunks = [new_fast.decode([tid]) for tid in token_ids]

    print(f"[{lang}]")
    print(f"  Text:   {phrase}")
    print(f"  Length: {len(token_ids)} tokens")
    print(f"  IDs:    {token_ids}")
    print(f"  Chunks: {chunks}\n")

--- Tokenization Comparison ---
[English]
  Text:   Hello, how are you doing today?
  Length: 8 tokens
  IDs:    [37254, 18, 1182, 450, 394, 4477, 2845, 37]
  Chunks: [' Hello', ',', ' how', ' are', ' you', ' doing', ' today', '?']

[French]
  Text:   Bonjour, comment allez-vous aujourd'hui ?
  Length: 17 tokens
  IDs:    [10860, 38019, 18, 9189, 9806, 96, 19, 92, 878, 214, 1843, 650, 74, 13, 15340, 79, 8412]
  Chunks: [' Bon', 'jour', ',', ' comment', ' alle', 'z', '-', 'v', 'ous', ' a', 'uj', 'our', 'd', "'", 'hu', 'i', ' ?']

[Kinyarwanda]
  Text:   Muraho, amakuru yawe uyumunsi?
  Length: 8 tokens
  IDs:    [290, 5572, 18, 1762, 5405, 11283, 7124, 37]
  Chunks: [' M', 'uraho', ',', ' amakuru', ' yawe', ' uy', 'umunsi', '?']



## 6 Vocabulary analysis 
Analyze the overlap between old and new vocabularies.

This helps understand:
- How many tokens are preserved (copied embeddings)
- How many tokens are new (need initialization)
- Examples of new Kinyarwanda-specific tokens


In [43]:
vocab_old = tokenizer_orig.get_vocab()  # token → old_id
vocab_new = new_fast.get_vocab()        # token → new_id

V_new = len(new_fast)


# Find overlap and new tokens
overlap_tokens = []
new_tokens = []

for tok_str, nid in vocab_new.items():
    if tok_str in vocab_old:
        overlap_tokens.append(tok_str)
    else:
        new_tokens.append(tok_str)

print(f"\n{'='*60}")
print(f"VOCABULARY ANALYSIS")
print(f"{'='*60}")
print(f"Original vocab size: {V_old:,}")
print(f"New vocab size:      {V_new:,}")
print(f"Overlapping tokens:  {len(overlap_tokens):,} ({len(overlap_tokens)/len(vocab_new)*100:.1f}%)")
print(f"New tokens:          {len(new_tokens):,} ({len(new_tokens)/len(vocab_new)*100:.1f}%)")
print(f"{'='*60}\n")

# INSPECTION: Show examples of new tokens (likely Kinyarwanda-specific)
print("📋 EXAMPLE NEW TOKENS (not in original vocab):")
print("-" * 60)
for i, tok in enumerate(new_tokens[:20]):  # Show first 20 new tokens
    print(f"  {i+1:2d}. '{tok}'")
print(f"  ... and {len(new_tokens) - 20} more")

# INSPECTION: Show examples of preserved tokens
print(f"\n📋 EXAMPLE PRESERVED TOKENS (in both vocabs):")
print("-" * 60)
for i, tok in enumerate(overlap_tokens[:20]):  # Show first 20 overlap tokens
    print(f"  {i+1:2d}. '{tok}'")
print(f"  ... and {len(overlap_tokens) - 20} more")

# INSPECTION: Check specific Kinyarwanda words
print(f"\n📋 KINYARWANDA TOKEN INSPECTION:")
print("-" * 60)
kin_examples = ["Muraho", "amakuru", "yawe", "umunsi", "neza"]
for word in kin_examples:
    # Check if word exists in new vocab
    in_new = word in vocab_new
    # Check how it's tokenized in new tokenizer
    new_ids = new_fast.encode(word, add_special_tokens=False)
    new_chunks = [new_fast.decode([id]) for id in new_ids]
    # Check how it was tokenized in old tokenizer
    old_ids = tokenizer_orig.encode(word, add_special_tokens=False)
    old_chunks = [tokenizer_orig.decode([id]) for id in old_ids]
    
    print(f"  Word: '{word}'")
    print(f"    In new vocab: {in_new}")
    print(f"    New tokenizer: {new_chunks} ({len(new_ids)} tokens)")
    print(f"    Old tokenizer: {old_chunks} ({len(old_ids)} tokens)")
    if in_new and word in vocab_old:
        print(f"    ✓ Embedding will be COPIED from original")
    elif in_new:
        print(f"    ⚡ Embedding will be INITIALIZED (avg_old)")
    print()


VOCABULARY ANALYSIS
Original vocab size: 262,144
New vocab size:      64,000
Overlapping tokens:  9,140 (14.3%)
New tokens:          54,860 (85.7%)

📋 EXAMPLE NEW TOKENS (not in original vocab):
------------------------------------------------------------
   1. 'Ġbahorana'
   2. 'leji'
   3. 'Ġdive'
   4. 'Ġretailer'
   5. 'Ġyasabiye'
   6. 'ĠUmucuruzi'
   7. 'imib'
   8. 'Ġamuha'
   9. 'ĠBefore'
  10. 'Ġreally'
  11. 'emuwe'
  12. 'Ġadequate'
  13. 'ishim'
  14. 'ĠBow'
  15. 'Ġayisumbuye'
  16. 'Ġzaho'
  17. 'abatanga'
  18. 'endeweho'
  19. 'Ġbac'
  20. 'Reduc'
  ... and 54840 more

📋 EXAMPLE PRESERVED TOKENS (in both vocabs):
------------------------------------------------------------
   1. 'Med'
   2. 'fers'
   3. 'oche'
   4. 'Technical'
   5. 'umbo'
   6. 'Gener'
   7. 'atility'
   8. 'PO'
   9. 'semb'
  10. 'ANI'
  11. 'anic'
  12. 'nex'
  13. 'Customer'
  14. 'Sal'
  15. 'FDA'
  16. 'wei'
  17. 'oned'
  18. 'iyon'
  19. 'shake'
  20. 'agit'
  ... and 9120 more

📋 KINYARWANDA 

## Notes of the new tokenizer 


- Ġ = Space. It means the word appears after a whitespace.
- It is **normal**. It appears because you used ByteLevel(add_prefix_space=True).
- the code handles it. The init_avg_old function explicitly strips Ġ and replaces it with a real space before looking up old embeddings, ensuring the remapping works correctly despite the difference between the new tokenizer (Ġ) and the original Gemma tokenizer 



## 7. Verify special tokens 

Special tokens (BOS, EOS, PAD, UNK, etc.) must be preserved exactly.

We verify that the NEW tokenizer loaded from disk has the required special tokens set.


In [46]:
print("[verify] Checking new tokenizer special tokens...")

required_tokens = ["bos_token", "eos_token", "pad_token"]
missing = []

for token_name in required_tokens:
    token_val = getattr(new_fast, token_name, None)
    token_id = getattr(new_fast, f"{token_name}_id", None)
    
    if token_val is None or token_id is None:
        missing.append(token_name)
        print(f"  ⚠️  WARNING: {token_name} is NOT set in new_fast!")
    else:
        print(f"  ✓ {token_name}: '{token_val}' (ID: {token_id})")

if missing:
    print("\n⚠️  CRITICAL: You must assign these special tokens to new_fast")
    print("   before remapping, or the model config will be invalid.")
    print("   Example: new_fast.pad_token = new_fast.eos_token")
else:
    print("\n✓ All special tokens are correctly set in new_fast.")
    print("✓ You can proceed to Embedding Initialization (Cell 6).")

[verify] Checking new tokenizer special tokens...
  ✓ bos_token: '<bos>' (ID: 0)
  ✓ eos_token: '<eos>' (ID: 2)
  ✓ pad_token: '<pad>' (ID: 4)

✓ All special tokens are correctly set in new_fast.
✓ You can proceed to Embedding Initialization (Cell 6).


## 7: Define Embedding Initialization Function (init_avg_old)


For NEW tokens (not in original vocab), we need to initialize embeddings.

The "avg_old" method:
1. Take the new token string (e.g., "Muraho")
2. Tokenize it with the ORIGINAL tokenizer → gets subword IDs
3. Look up embeddings for each subword in ORIGINAL embedding matrix (E0)
4. Compute weighted average (longer subwords get more weight)
5. Use this as the initial embedding for the new token

WHY? This gives new tokens a reasonable starting point based on their
constituent parts, rather than random initialization.

EXAMPLE:
  New token: "▁Muraho"
  Old tokenizer splits: ["▁Mur", "aho"]
  New embedding = avg(embedding["▁Mur"], embedding["aho"])



In [47]:

def init_avg_old(token: str, debug: bool = False) -> torch.Tensor:
    """
    Initialize embedding for a new token by averaging subword embeddings from original model.
    
    Args:
        token: The token string (e.g., "▁Muraho")
        debug: If True, print decomposition details
    
    Returns:
        torch.Tensor: Initial embedding vector (hidden_dim,)
    """
    text = token
    
    # Handle space markers (Ġ = GPT2-style, ▁ = SentencePiece-style)
    if token.startswith("Ġ") or token.startswith("▁"):
        text = " " + token[1:]
        if debug:
            print(f"    [debug] Converted '{token}' → '{text}'")
    
    # Tokenize with ORIGINAL tokenizer (no special tokens)
    ids = tokenizer_orig(text, add_special_tokens=False)["input_ids"]
    
    if debug:
        print(f"    [debug] Old tokenizer IDs: {ids}")
        print(f"    [debug] Old tokenizer chunks: {[tokenizer_orig.decode([id]) for id in ids]}")
    
    # If tokenization failed, return small random vector
    if not ids:
        if debug:
            print(f"    [debug] ⚠️ No IDs returned, using random initialization")
        return torch.empty((hidden,), dtype=torch.float32).normal_(mean=0.0, std=0.02)
    
    # Compute weighted average of subword embeddings
    total = torch.zeros((hidden,), dtype=torch.float32)
    wsum = 0.0
    
    for oid in ids:
        if 0 <= oid < V_old:
            piece = tokenizer_orig.convert_ids_to_tokens(oid)
            w = max(1, len(piece))  # Weight by string length
            total += E0[oid] * w
            wsum += w
            if debug:
                print(f"    [debug]   Subword '{piece}' (ID={oid}, weight={w})")
    
    # If no valid pieces, return random vector
    if wsum == 0:
        if debug:
            print(f"    [debug] ⚠️ Zero weight sum, using random initialization")
        return torch.empty((hidden,), dtype=torch.float32).normal_(mean=0.0, std=0.02)
    
    # Return weighted average
    result = total / wsum
    if debug:
        print(f"    [debug] ✓ Final embedding: mean={result.mean().item():.4f}, std={result.std().item():.4f}")
    
    return result


# TEST the function with a sample token
print("\n🧪 TESTING init_avg_old FUNCTION:")
print("-" * 60)
test_token = "▁Muraho"  # Example Kinyarwanda token
print(f"Token: '{test_token}'")
print(f"In original vocab: {test_token in vocab_old}")
print(f"\nDecomposition:")
vec = init_avg_old(test_token, debug=True)
print(f"\n✓ Result vector shape: {vec.shape}")
print(f"✓ Result vector dtype: {vec.dtype}")


🧪 TESTING init_avg_old FUNCTION:
------------------------------------------------------------
Token: '▁Muraho'
In original vocab: False

Decomposition:
    [debug] Converted '▁Muraho' → ' Muraho'
    [debug] Old tokenizer IDs: [10871, 21780]
    [debug] Old tokenizer chunks: [' Mur', 'aho']
    [debug]   Subword '▁Mur' (ID=10871, weight=4)
    [debug]   Subword 'aho' (ID=21780, weight=3)
    [debug] ✓ Final embedding: mean=0.0007, std=0.0323

✓ Result vector shape: torch.Size([640])
✓ Result vector dtype: torch.float32


## 8 Create New Embedding Matrix (Remapping + Initialization)

This is the CORE step. We create a new embedding matrix (E1) with shape:
  (V_new, hidden_dim)

For each token in the NEW vocabulary:
1. If token exists in OLD vocab → COPY the original embedding
2. If token is NEW → INITIALIZE using init_avg_old (or random/zeros)

This preserves knowledge for overlapping tokens while giving new tokens
a reasonable starting point.



In [49]:

## define the merging method 

INIT_METHOD = "avg_old"



print("[embed] Creating new embedding matrix...")
print(f"  Target shape: ({V_new:,}, {hidden})")
print(f"  Target dtype: torch.bfloat16")
print(f"  Initialization method: {INIT_METHOD}")
print()

# Initialize new embedding matrix
E1 = torch.empty((V_new, hidden), dtype=torch.bfloat16)

# Counters for tracking
copied = 0      # Tokens copied from original
inited = 0      # Tokens initialized (new)
random_init = 0 # Tokens that fell back to random

# Progress tracking
progress_interval = V_new // 10

# Main remapping loop
for idx, (tok_str, nid) in enumerate(vocab_new.items()):
    # Show progress every 10%
    if idx % progress_interval == 0 and idx > 0:
        pct = (idx / V_new) * 100
        print(f"  Progress: {idx:,}/{V_new:,} ({pct:.0f}%) - copied={copied}, inited={inited}")
    
    # Check if token exists in original vocabulary
    oid = vocab_old.get(tok_str, None)
    
    if oid is not None and 0 <= oid < V_old:
        # CASE 1: Token exists in original vocab → COPY embedding
        E1[nid] = E0[oid].to(torch.bfloat16)
        copied += 1
    else:
        # CASE 2: New token → INITIALIZE embedding
        if INIT_METHOD == "avg_old":
            vec = init_avg_old(tok_str)
            E1[nid] = vec.to(torch.bfloat16)
        elif INIT_METHOD == "zeros":
            E1[nid] = torch.zeros((hidden,), dtype=torch.bfloat16)
        else:  # "random"
            tmp = torch.empty((hidden,), dtype=torch.float32).normal_(mean=0.0, std=0.02)
            E1[nid] = tmp.to(torch.bfloat16)
            random_init += 1
        inited += 1

print(f"\n{'='*60}")
print(f"EMBEDDING MATRIX CREATION COMPLETE")
print(f"{'='*60}")
print(f"Total tokens:     {V_new:,}")
print(f"Copied (old):     {copied:,} ({copied/V_new*100:.1f}%)")
print(f"Initialized (new):{inited:,} ({inited/V_new*100:.1f}%)")
print(f"  - Random fallback: {random_init:,}")
print(f"{'='*60}\n")

# INSPECTION: Verify some specific tokens
print("📋 EMBEDDING VERIFICATION:")
print("-" * 60)

# Check a preserved token (e.g., "the")
if "the" in vocab_new:
    nid = vocab_new["the"]
    oid = vocab_old.get("the")
    if oid:
        match = torch.allclose(E1[nid].to(torch.float32), E0[oid].to(torch.float32), atol=1e-5)
        print(f"✓ Token 'the': copied={match}, old_id={oid}, new_id={nid}")

# Check a new token (e.g., Kinyarwanda word)
for tok in ["▁Muraho", "▁amakuru", "▁neza"]:
    if tok in vocab_new:
        nid = vocab_new[tok]
        in_old = tok in vocab_old
        vec_mean = E1[nid].to(torch.float32).mean().item()
        vec_std = E1[nid].to(torch.float32).std().item()
        print(f"✓ Token '{tok}': in_old={in_old}, new_id={nid}, mean={vec_mean:.4f}, std={vec_std:.4f}")

[embed] Creating new embedding matrix...
  Target shape: (64,000, 640)
  Target dtype: torch.bfloat16
  Initialization method: avg_old

  Progress: 6,400/64,000 (10%) - copied=915, inited=5485
  Progress: 12,800/64,000 (20%) - copied=1822, inited=10978
  Progress: 19,200/64,000 (30%) - copied=2753, inited=16447
  Progress: 25,600/64,000 (40%) - copied=3679, inited=21921
  Progress: 32,000/64,000 (50%) - copied=4563, inited=27437
  Progress: 38,400/64,000 (60%) - copied=5422, inited=32978
  Progress: 44,800/64,000 (70%) - copied=6375, inited=38425
  Progress: 51,200/64,000 (80%) - copied=7302, inited=43898
  Progress: 57,600/64,000 (90%) - copied=8219, inited=49381

EMBEDDING MATRIX CREATION COMPLETE
Total tokens:     64,000
Copied (old):     9,139 (14.3%)
Initialized (new):54,861 (85.7%)
  - Random fallback: 0

📋 EMBEDDING VERIFICATION:
------------------------------------------------------------
✓ Token 'the': copied=True, old_id=1437, new_id=2919


## 9 . Resize Model and Assign New Embeddings

Now we modify the original model to use the new embedding matrix.

Steps:
1. Resize token embeddings layer to V_new
2. Convert model to bfloat16 (Gemma 3 native precision)
3. Copy our new embedding matrix (E1) into the model
4. Tie input/output embeddings (Gemma 3 uses tied embeddings)
5. Update model config with new special token IDs

IMPORTANT: All operations are done with torch.no_grad() since we're
not training yet - just setting up the model structure.


In [50]:

print("[model] Resizing and configuring model...")

model = model_orig  # Work with the loaded model

with torch.no_grad():
    # 1. Resize token embeddings to new vocabulary size
    model.resize_token_embeddings(V_new)
    print(f"✓ Resized embeddings to {V_new:,} tokens")
    
    # 2. Convert model to bfloat16
    model.to(dtype=torch.bfloat16)
    print(f"✓ Converted model to bfloat16")
    
    # 3. Copy our new embedding matrix into the model
    model.get_input_embeddings().weight[:] = E1.to(model.get_input_embeddings().weight.device)
    print(f"✓ Assigned new embedding matrix")
    
    # 4. Tie input/output embeddings (Gemma 3 requirement)
    model.config.tie_word_embeddings = True
    if hasattr(model, "tie_weights"):
        model.tie_weights()
    print(f"✓ Tied input/output embeddings")
    
    # 5. Update config with new special token IDs
    model.config.eos_token_id = new_fast.eos_token_id
    model.config.bos_token_id = new_fast.bos_token_id
    model.config.pad_token_id = new_fast.pad_token_id
    model.config.vocab_size = V_new
    print(f"✓ Updated model config (BOS={model.config.bos_token_id}, EOS={model.config.eos_token_id})")

print(f"\n✓ Model ready for saving")

[model] Resizing and configuring model...
✓ Resized embeddings to 64,000 tokens
✓ Converted model to bfloat16
✓ Assigned new embedding matrix
✓ Tied input/output embeddings
✓ Updated model config (BOS=0, EOS=2)

✓ Model ready for saving


## 10 Save Generation Configuration

The generation_config.json controls how the model generates text
(max_tokens, temperature, special tokens, etc.).

We download the original and update it with our new special token IDs.
This ensures generation works correctly with the new tokenizer.

In [53]:

print("[config] Saving generation configuration...")

try:
    # Download original generation_config.json
    gen_path = hf_hub_download(MODEL_ID, "generation_config.json")
    gen = json.loads(Path(gen_path).read_text(encoding="utf-8"))
    
    # Update with new special token IDs
    gen.update({
        "eos_token_id": new_fast.eos_token_id,
        "bos_token_id": new_fast.bos_token_id,
        "pad_token_id": new_fast.pad_token_id,
        "vocab_size": V_new
    })
    
    # Save updated config
    (OUT_DIR_NewModel / "generation_config.json").write_text(json.dumps(gen, indent=2), encoding="utf-8")
    print(f"✓ Generation config saved to {OUT_DIR_NewModel / 'generation_config.json'}")
    
except Exception as e:
    print(f"⚠️ Warning: Could not save generation_config.json: {e}")
    print("  Model will still work, but generation defaults may differ")

[config] Saving generation configuration...
✓ Generation config saved to gemma3-270M-kin-en-64k_model/generation_config.json


## 11. Save Adapted Model to Disk


Save the adapted model with new tokenizer and embeddings.

The model is saved in Hugging Face format, which means:
- config.json (model architecture)
- model.safetensors (weights in safe tensor format)
- generation_config.json (generation parameters)
- tokenizer files (from new_fast)

After saving, you can load this model like any other HF model.



In [56]:

print(f"[save] Saving model to {OUT_DIR_NewModel}...")

# Save model weights and config
model.save_pretrained(OUT_DIR_NewModel, safe_serialization=True)
print(f"✓ Model weights saved")

# Save the new tokenizer (should already be saved, but ensure consistency)
new_fast.save_pretrained(OUT_DIR_NewModel)
print(f"✓ Tokenizer saved")

# Check file sizes
total_size = sum(f.stat().st_size for f in OUT_DIR_NewModel.glob("*") if f.is_file())
print(f"\n{'='*60}")
print(f"MODEL SAVED SUCCESSFULLY")
print(f"{'='*60}")
print(f"Output directory: {OUT_DIR_NewModel.absolute()}")
print(f"Total size: {total_size / 1024 / 1024:.1f} MB")
print(f"Vocabulary size: {V_new:,} tokens")
print(f"Parameter count: ~{V_new * hidden / 1e6:.1f}M (embeddings only)")
print(f"{'='*60}\n")

[save] Saving model to gemma3-270M-kin-en-64k_model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Model weights saved
✓ Tokenizer saved

MODEL SAVED SUCCESSFULLY
Output directory: /home/xmikelinux/__Projects/01_LLM_vocabReduction/gemma3-270M-kin-en-64k_model
Total size: 273.9 MB
Vocabulary size: 64,000 tokens
Parameter count: ~41.0M (embeddings only)



## 12. Validation Tests - Verify Model Works

Before using the model, we must verify:
1. Tokenizer loads correctly
2. Model loads correctly
3. Tokenization works for both Kinyarwanda and English
4. Generation produces coherent output (basic sanity check)

This catches configuration errors early.


In [57]:
print("\n{'='*60}")
print("VALIDATION TESTS")
print(f"{'='*60}\n")

# Load the saved model and tokenizer
print("[test] Loading saved model and tokenizer...")
tok_new = AutoTokenizer.from_pretrained(OUT_DIR_NewModel, use_fast=True)
m_eval = AutoModelForCausalLM.from_pretrained(
    OUT_DIR_NewModel, 
    trust_remote_code=True,
    torch_dtype=torch.bfloat16
).eval()

# Move to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
m_eval = m_eval.to(device)
print(f"✓ Loaded on {device}")

# TEST 1: Vocabulary size match
print(f"\n📋 TEST 1: Vocabulary Size")
print(f"  Tokenizer vocab: {len(tok_new):,}")
print(f"  Model embeddings: {m_eval.get_input_embeddings().weight.shape[0]:,}")
assert len(tok_new) == m_eval.get_input_embeddings().weight.shape[0], "❌ MISMATCH!"
print(f"  ✓ PASS\n")

# TEST 2: Kinyarwanda Tokenization
print(f"📋 TEST 2: Kinyarwanda Tokenization")
kin_text = "Muraho, amakuru yawe uyumunsi?"
kin_ids = tok_new(kin_text, return_tensors="pt").to(device)
print(f"  Input: '{kin_text}'")
print(f"  Tokens: {kin_ids['input_ids'].shape[1]}")
print(f"  Chunks: {[tok_new.decode([id]) for id in kin_ids['input_ids'][0]]}")
print(f"  ✓ PASS\n")

# TEST 3: English Tokenization
print(f"📋 TEST 3: English Tokenization")
en_text = "Hello, how are you today?"
en_ids = tok_new(en_text, return_tensors="pt").to(device)
print(f"  Input: '{en_text}'")
print(f"  Tokens: {en_ids['input_ids'].shape[1]}")
print(f"  Chunks: {[tok_new.decode([id]) for id in en_ids['input_ids'][0]]}")
print(f"  ✓ PASS\n")

# TEST 4: Basic Generation (Kinyarwanda)
print(f"📋 TEST 4: Basic Generation (Kinyarwanda)")
kin_prompt = "Muraho, amakuru"
kin_enc = tok_new(kin_prompt, return_tensors="pt").to(device)
with torch.no_grad():
    kin_out = m_eval.generate(
        **kin_enc, 
        max_new_tokens=15, 
        pad_token_id=tok_new.eos_token_id,
        do_sample=False
    )
kin_result = tok_new.decode(kin_out[0], skip_special_tokens=True)
print(f"  Prompt: '{kin_prompt}'")
print(f"  Output: '{kin_result}'")
print(f"  ✓ PASS\n")

# TEST 5: Basic Generation (English)
print(f"📋 TEST 5: Basic Generation (English)")
en_prompt = "Hello, how are"
en_enc = tok_new(en_prompt, return_tensors="pt").to(device)
with torch.no_grad():
    en_out = m_eval.generate(
        **en_enc, 
        max_new_tokens=15, 
        pad_token_id=tok_new.eos_token_id,
        do_sample=False
    )
en_result = tok_new.decode(en_out[0], skip_special_tokens=True)
print(f"  Prompt: '{en_prompt}'")
print(f"  Output: '{en_result}'")
print(f"  ✓ PASS\n")

print(f"{'='*60}")
print(f"ALL VALIDATION TESTS PASSED ✓")
print(f"{'='*60}")
print(f"\n⚠️  IMPORTANT: Model is NOT yet ready for production!")
print(f"   You must run CONTINUOUS PRE-TRAINING (CPT) to recalibrate")
print(f"   the model to the new tokenization boundaries.")


`torch_dtype` is deprecated! Use `dtype` instead!



{'='*60}
VALIDATION TESTS

[test] Loading saved model and tokenizer...


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✓ Loaded on cuda

📋 TEST 1: Vocabulary Size
  Tokenizer vocab: 64,000
  Model embeddings: 64,000
  ✓ PASS

📋 TEST 2: Kinyarwanda Tokenization
  Input: 'Muraho, amakuru yawe uyumunsi?'
  Tokens: 8
  Chunks: [' M', 'uraho', ',', ' amakuru', ' yawe', ' uy', 'umunsi', '?']
  ✓ PASS

📋 TEST 3: English Tokenization
  Input: 'Hello, how are you today?'
  Tokens: 7
  Chunks: [' Hello', ',', ' how', ' are', ' you', ' today', '?']
  ✓ PASS

📋 TEST 4: Basic Generation (Kinyarwanda)
  Prompt: 'Muraho, amakuru'
  Output: ' Muraho, amakuru, am, am, am, am, am, am, am,'
  ✓ PASS

📋 TEST 5: Basic Generation (English)
  Prompt: 'Hello, how are'
  Output: ' Hello, how are you all, how are you all, how are you all, how are'
  ✓ PASS

ALL VALIDATION TESTS PASSED ✓

⚠️  IMPORTANT: Model is NOT yet ready for production!
   You must run CONTINUOUS PRE-TRAINING (CPT) to recalibrate
   the model to the new tokenization boundaries.
   See File 1 (Benjamin Marie) for CPT instructions.



In [66]:
print(f"📋 TEST 4: Basic Generation (Kinyarwanda)")
kin_prompt = "le monde  "
kin_enc = tok_new(kin_prompt, return_tensors="pt").to(device)
with torch.no_grad():
    kin_out = m_eval.generate(
        **kin_enc, 
        max_new_tokens=15, 
        pad_token_id=tok_new.eos_token_id,
        do_sample=False
    )
kin_result = tok_new.decode(kin_out[0], skip_special_tokens=True)
print(f"  Prompt: '{kin_prompt}'")
print(f"  Output: '{kin_result}'")
print(f"  ✓ PASS\n")

📋 TEST 4: Basic Generation (Kinyarwanda)
  Prompt: 'le monde  '
  Output: ' le monde                 '
  ✓ PASS



In [63]:
print(f"📋 TEST 5: Basic Generation (English)")
en_prompt = "Hello, how are"
en_enc = tok_new(en_prompt, return_tensors="pt").to(device)
with torch.no_grad():
    en_out = m_eval.generate(
        **en_enc, 
        max_new_tokens=15, 
        pad_token_id=tok_new.eos_token_id,
        do_sample=False
    )

en_result = tok_new.decode(en_out[0], skip_special_tokens=True)
print(f"  Prompt: '{en_prompt}'")
print(f"  Output: '{en_result}'")
print(f"  ✓ PASS\n")    

📋 TEST 5: Basic Generation (English)
  Prompt: 'Hello, how are'
  Output: ' Hello, how are you all, how are you all, how are you all, how are'
  ✓ PASS

